# 007 · Dilution model (not started)

**The question.** Will this company issue many new shares soon?

**The answer.** `dilution`: 1 if shares outstanding (split-adjusted, as first filed) rise by 10% or more within 182 days, else 0. About 10% of rows are 1.

**The baseline to beat.** the base rate: the same chance (about 10%) for every row.

**Measures.** AUC, log loss, Brier score (always saying the base rate scores 0.0898), and lift in the top tenth.

**Status.** Not started: headings and instructions only. The model is built, run and tuned in
this notebook in feature 007
([docs/plan.md](../docs/plan.md), issue in the roadmap).

### What the answer means (read this first)

Think of an exam: the 96 **features** are the questions (what was known about a company on a
month-end: cash, losses, recent filings, price...), and **`dilution`** is the answer key (what
happened next). The model learns from rows *with* the key and is judged on rows whose key it never
saw.

- **The answer:** 1 if the company's shares outstanding (as filed on its 10-Q and 10-K covers,
  split-adjusted) rose **10% or more within the next 182 days** (6 months), 0 if not, missing if
  no new count was filed. Example: Company X on 31 March 2021 had 50 million shares (last count
  filed by then); the last count filed by 29 September 2021 says 58 million: +16%, so 1. With 52
  million (+4%), 0. With no new filing, missing (the row is not used).
- **Why 10%, not "any rise":** share counts creep up almost everywhere (employee stock: the median
  company +0.6% a year; two thirds of rows rise at all), so "any rise" would flag most healthy
  companies. What hurts shareholders is a share sale, which shows as a jump; above 10% the typical
  jump is +27%. The 10% is pre-registered (`config/risk_params.yaml` v5) and stays (decided 30
  Oct; an "offering event" question is roadmap 024).
- **Filings vs share counts:** filings (offering prospectuses, shelf registrations, 8-K 3.02
  unregistered sales) come early and are **features**: warning signs (after an offering filed in
  the past year, 21% of rows dilute 10%+ next; after an 8-K unregistered sale 27%; with none of
  these warning signs 2.3%). The share count is
  late but true, so it is the **answer**. The app shows the chance with these warning signs beside
  it.
- **Which rows and columns:** fit only on `dilution.parquet`, rows with `eligible_dilution` true
  and `dilution` present (409,576; base rate 9.6%); columns: the 96 features, never `dilution`
  or `window_end`. Save the feature list with the model.
- **How error is measured, in three stages, always on answers the model has not seen:** (1) the
  walk-forward test years 2019–2022 inside `dilution.parquet` (train on earlier years, score the
  year: this chooses the model); (2) the holdout, once (feature 010): the model predicts the
  2023+ month-end rows of `feature_snapshot.parquet`, which have **features but no answers** on
  purpose; the guarded command then computes their real answers from the share counts filed since
  and scores the predictions; (3) live (feature 009): each day's forecast is stored and scored
  six months later. `latest.parquet` is each company's newest row, for today's forecast.

### Rules for every model notebook

- **Spec first.** Before filling this notebook, run `/speckit-specify` for this feature with
  `SPECIFY_FEATURE_DIRECTORY=specs/007-dilution-model` on branch `007-dilution-model` (README, "Working
  together").
- **This notebook is where the model is built.** Every level is created, run and tuned here,
  with its tables and charts, start to finish by one command, so the researcher can read how the
  choice was made. Once the researcher is happy with the chosen model, its code moves into
  `src/sentinel/models/` (training, scoring, calibration) so that `sentinel update` scores every
  company each night without a notebook and the repository runs from a clone; this notebook then
  imports that code and must give the same numbers (docs/plan.md, "Notebooks and code").
- **Data.** Read only `data/exports/ml/dilution.parquet` through `sentinel.eda.load("dilution")`, which keeps
  the development rows (dated 2022 or earlier). Learn and test only on rows where this question's
  flag `eligible_dilution` is true and the answer is present (V2-26, V2-27). Never read `data/exports/reference/` (answers for 2023 on) and
  never compute anything for 2023 or later: that is the sealed holdout, tested once by feature 010
  (constitution IV).
- **Never use as features:** `dilution`, `window_end`, `cik`, `as_of` and the flags.
- **Never a random train/test split.** Rows of one company over time are not independent, and a
  random split lets the model learn from the future. The only split is walk-forward by year:
- **Evaluation is fixed** (decision D60, `config/risk_params.yaml` v5, never edited): yearly
  walk-forward test years 2019, 2020, 2021, 2022; for each, train only on rows whose answer window
  ended before 1 January of that year (`sentinel.eda.purge_table` counts them); four levels:
  naive baseline → rule → classic model → LightGBM; choose the **simplest level within one
  standard error of the best**; report every test year, not only the average.
- **Show forecasts as a chance in five bands, each with its real dilution rate on the test years** with their hit rates on the test years (D62), never as a bare
  number.
- **Reproducible:** sort rows by `cik`, `as_of` before modelling; fixed seeds (`SEED = 0`); log
  each run in MLflow with the git commit and the export date. Two runs must give identical
  predictions.
- **Private data:** outputs are stripped on commit (nbstripout); written conclusions quote
  aggregates only, never one company's prices or returns (Tiingo terms). Never write to `data/`.
- **Run it** start to finish under the memory cap:
  `systemd-run --user --scope -p MemoryMax=5G -p MemorySwapMax=0 uv run jupyter nbconvert --to notebook --execute --inplace notebooks/007-dilution-model.ipynb`

### Two kinds of test, and the final one is not in this notebook

1. **Test years 2019–2022 (here).** For each test year, train on the earlier years only (with
   the purge) and score that year. This is where the four levels are compared and **one is
   chosen**. Because the choice is made by looking at these results, they flatter the winner a
   little.
2. **The holdout: answers whose windows start in 2023 or later (not here, not in any notebook).**
   It answers the only question that matters in the end: does the model chosen here work on data
   nobody looked at while choosing it? That works only if it is used **once**.

**How it stays hidden.** The training file has no answers after 2022: the pipeline does not even
compute them (the label code stops at windows ending 31 December 2022). This notebook must not
read the private reference copies in `data/exports/reference/` (feature 002), which hold them.

**How the final test is run (feature 010, 16 November).**

1. This feature ends with the chosen model **frozen** in `src/sentinel/models/`: its settings,
   its training rows (to 2022), its seed, the git commit and the export date, logged in MLflow.
   The report (008) is written from the 2019–2022 results.
2. The researcher then runs **one guarded command** (built in 010). It checks that the models are
   frozen and that no holdout run exists yet, computes the 2023+ answers, scores the frozen
   models, and writes the results with a log entry. A second run is refused: the log is the proof
   that the test happened once.
3. The results go into the report as they came out. The model is not changed afterwards
   (constitution IV). Later refits (021) may use the 2023+ answers, as a logged step.

A command, not a notebook, because a notebook can be re-run with one click and nothing records
how often it was looked at.

### Make the model holdout-ready (so the one test runs without a hitch)

The holdout rows come from the same feature table as the training rows (`feature_snapshot`
from 2023 on: same columns, same order, same types, checked on 30 October) and their answers from
the same label code with a later end date. What can still break is the model itself, so:

1. **Select features by name, never by position**: save the list of feature columns with the
   model and use `rows[features]` when scoring.
2. **Every preprocessing step lives inside the model** (cut points, fill values, missing flags,
   scaling), fitted on the training rows only and saved with it, never recomputed on new rows.
3. **Save one object** (in MLflow): the fitted preprocessing and model, the feature list, the
   eligibility flag it uses (`{elig}`), the seed, the git commit and the export date.
4. **Dry run before 16 November**: load the saved model and score `latest.parquet` and the 2023+
   rows of `feature_snapshot.parquet` (features only, no answers): it must run, give one
   prediction per eligible row, no unexpected missing values, values in a plausible range. This
   looks at no outcome, so it does not spend the holdout.
5. **Scoring is a function** `predict(model, rows) -> predictions` in `src/sentinel/models/`,
   the same one used for the test years, the dry run, the holdout and the daily scores.

### What the EDA found for this model (notebooks/004-eda-dilution.ipynb, Findings)

- **Rule level ready-made:** "an offering filed in the last 12 months" (`n_offering_12m > 0`):
  AUC 0.70–0.73 in every test year; dilution 21.8% if flagged, 3.8% if not. A model must clearly
  beat it.
- Strongest features: offerings and shelf filings, share growth over 12 months, interest
  coverage (low or missing), R&D intensity, stock-based pay to revenue, volatility, low price,
  small size, loss-making. Macro and short volume add nothing (drop them from the classic
  model).
- Missing values carry meaning (no interest coverage often means a small company without debt):
  add "was missing" flags.
- **Foreign filers are left out of this question (V2-27, decided 30 Oct):** train, test and
  score only rows with `foreign_filer == False`; 78% of their rows have no answer and the rest
  are not typical.
- Rows with a stock split in the window dilute 25% vs 9.7%: open question V2-25 (named cases
  first). Impossible share counts (V2-23) can make wrong answers.

**Data cleaning** (V2-33): the pipeline's cleaning is frozen at V2-32 (foreign market values,
share-count typos, insider value typos, placeholder prices, splits in the dilution answer, Tiingo
instruments; `docs/decisions.md`). Any further cleaning is part of **this model**: cut extreme
values at the 1st and 99th percentiles of the training rows, drop rows with a known doubt, add
"was missing" flags; write it here, explain why, and move it into `src/sentinel/models/` with the
model so the daily scoring cleans the same way. Do not rebuild the database for small fixes.

## 1. Setup and data

*To do:* Load the development rows, keep eligible rows with an answer, list the features to use (the EDA's advice below), and print rows and companies per test year.

In [ ]:
import subprocess
import warnings
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import lightgbm as lgb
import matplotlib.pyplot as plt
import mlflow
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import brier_score_loss, log_loss, roc_auc_score
from sklearn.preprocessing import StandardScaler
import xgboost as xgb

from sentinel import config, eda

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 120)

SEED = 0
TEST_YEARS = eda.TEST_YEARS  # [2019, 2020, 2021, 2022]

PATHS = config.paths()
EXPORT_FILE = eda.EXPORT / "dilution.parquet"
EXPORT_DATE = datetime.fromtimestamp(EXPORT_FILE.stat().st_mtime, tz=timezone.utc).strftime("%Y-%m-%d")
GIT_COMMIT = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], cwd=PATHS.root, text=True).strip()

# 1. Load development rows (2012-2022) with eligible_dilution and non-null dilution answer
raw_df = eda.load("dilution")
df = raw_df[raw_df["eligible_dilution"] & raw_df["dilution"].notna()].copy()
pre_len = len(df)

# 2. Exclude the 46 split-read-as-dilution false-positive cases identified in validation (V2-25)
split_cases = pd.read_csv(PATHS.exports / "share" / "dilution_split_46_cases.csv")
bad_pairs = set(zip(split_cases["cik"].astype(str), pd.to_datetime(split_cases["as_of"]).dt.date))
df = df[~df.apply(lambda r: (str(r["cik"]), r["as_of"].date()) in bad_pairs, axis=1)].copy().reset_index(drop=True)
print(f"Loaded {len(df):,} eligible development rows with answer (excluded {pre_len - len(df)} split false positives).")

# 3. Features: Drop insider and macro families based on the teammate's 5-seed ablation check
all_features = eda.features(df)
FEATURE_COLS = [c for c in all_features if eda.family(c) not in ("insider", "macro")]
print(f"Feature count: {len(FEATURE_COLS)} features (dropped {len(all_features) - len(FEATURE_COLS)} insider/macro features).")
assert "cum_split_after" not in FEATURE_COLS, "cum_split_after must never be used as a feature!"

# Summary statistics
print(f"Development base rate (overall): {df['dilution'].mean():.4f}")
for y in TEST_YEARS:
    sub = df[df["as_of"].dt.year == y]
    print(f"  Test year {y}: {len(sub):,} rows, {sub['cik'].nunique():,} companies, base rate = {sub['dilution'].mean():.4f}")

## 2. Walk-forward folds

*To do:* Build the four folds (test years 2019–2022) with the purge; print train and test rows per fold.

In [ ]:
# Four yearly walk-forward folds 2019-2022 with purge
FOLDS = {}
for y in TEST_YEARS:
    start = pd.Timestamp(year=y, month=1, day=1)
    train_mask = (df["as_of"] < start) & (df["window_end"] < start)
    test_mask = df["as_of"].dt.year == y
    FOLDS[y] = {
        "train": df[train_mask].sort_values(["cik", "as_of"]).reset_index(drop=True),
        "test": df[test_mask].sort_values(["cik", "as_of"]).reset_index(drop=True),
    }

print("Walk-forward purge table:")
display(eda.purge_table(df, "window_end"))

## 3. Level 0: the naive baseline

*To do:* Score the baseline on each test year with the chosen measures.

In [ ]:
def score_classifier_predictions(y_true: np.ndarray, y_prob: np.ndarray) -> dict[str, float]:
    eps = 1e-15
    p_clipped = np.clip(y_prob, eps, 1 - eps)
    auc = float(roc_auc_score(y_true, y_prob)) if len(np.unique(y_true)) > 1 else 0.5
    ll = float(log_loss(y_true, p_clipped))
    bs = float(brier_score_loss(y_true, y_prob))
    # Top decile lift: hit rate in top 10% predicted / overall base rate
    k = max(int(np.ceil(len(y_true) * 0.10)), 1)
    top_idx = np.argsort(y_prob)[-k:]
    lift = float((y_true[top_idx].mean()) / (y_true.mean() + eps))
    return {"auc": auc, "log_loss": ll, "brier": bs, "top_decile_lift": lift}

def summarize_scores(yearly_scores: dict[int, dict[str, float]]) -> pd.DataFrame:
    tab = pd.DataFrame(yearly_scores).T
    tab.index.name = "test_year"
    mean_s = tab.mean()
    se_s = tab.std(ddof=1) / np.sqrt(len(tab))
    out = pd.concat([tab, pd.DataFrame({"mean": mean_s, "se": se_s}).T])
    return out

level0_yearly = {}
for y, fold in FOLDS.items():
    tr, te = fold["train"], fold["test"]
    p_base = float(tr["dilution"].mean())
    pred_probs = np.full(len(te), p_base)
    level0_yearly[y] = score_classifier_predictions(te["dilution"].to_numpy(), pred_probs)

print("Level 0 — Naive baseline (historical base rate):")
display(summarize_scores(level0_yearly).round(4))

## 4. Level 1: a rule

*To do:* A simple, explainable rule (see the EDA's suggestion below); fit only on each fold's training rows.

In [ ]:
# Level 1 Rule: Recent share offering filings (n_offering_12m > 0)
# A company that filed an equity offering in the past 12 months is far more likely to dilute
offering_col = "n_offering_12m" if "n_offering_12m" in df.columns else "n_offering_filed_12m"

level1_yearly = {}
for y, fold in FOLDS.items():
    tr, te = fold["train"], fold["test"]
    flag_tr = (tr[offering_col] > 0).to_numpy()
    flag_te = (te[offering_col] > 0).to_numpy()
    
    # Estimate conditional hit rate from training fold only
    p_flagged = float(tr.loc[flag_tr, "dilution"].mean())
    p_unflagged = float(tr.loc[~flag_tr, "dilution"].mean())
    
    pred_probs = np.where(flag_te, p_flagged, p_unflagged)
    level1_yearly[y] = score_classifier_predictions(te["dilution"].to_numpy(), pred_probs)

print(f"Level 1 — Rule ({offering_col} > 0):")
display(summarize_scores(level1_yearly).round(4))

## 5. Level 2: a classic model

*To do:* A linear or logistic model: one feature of each redundant pair, ratios cut at their 1st and 99th percentiles (on training rows), missing values filled with the training median plus a 'was missing' flag, standardised.

In [ ]:
def fit_classic_logistic(tr: pd.DataFrame, feature_cols: list[str]) -> tuple[LogisticRegression, StandardScaler, dict, dict]:
    clip_lo, clip_hi, medians = {}, {}, {}
    X_parts = []
    
    for c in feature_cols:
        s = pd.to_numeric(tr[c], errors="coerce").astype(float)
        finite = s.dropna()
        q01, q50, q99 = (finite.quantile([0.01, 0.50, 0.99])) if len(finite) else (0.0, 0.0, 0.0)
        clip_lo[c], medians[c], clip_hi[c] = float(q01), float(q50), float(q99)
        
        # Continuous clipped and imputed
        val = s.clip(lower=clip_lo[c], upper=clip_hi[c]).fillna(medians[c]).to_numpy(dtype=np.float32)
        # Missing indicator
        miss = s.isna().to_numpy(dtype=np.float32)
        X_parts.extend([val, miss])
        
    X_mat = np.column_stack(X_parts)
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_mat)
    
    model = LogisticRegression(C=0.1, penalty="l2", max_iter=1000, random_state=SEED)
    model.fit(X_scaled, tr["dilution"].to_numpy(dtype=int))
    return model, scaler, clip_lo, clip_hi, medians

def predict_classic_logistic(model, scaler, clip_lo, clip_hi, medians, frame: pd.DataFrame, feature_cols: list[str]) -> np.ndarray:
    X_parts = []
    for c in feature_cols:
        s = pd.to_numeric(frame[c], errors="coerce").astype(float)
        val = s.clip(lower=clip_lo[c], upper=clip_hi[c]).fillna(medians[c]).to_numpy(dtype=np.float32)
        miss = s.isna().to_numpy(dtype=np.float32)
        X_parts.extend([val, miss])
    X_mat = np.column_stack(X_parts)
    X_scaled = scaler.transform(X_mat)
    return model.predict_proba(X_scaled)[:, 1]

level2_yearly = {}
for y, fold in FOLDS.items():
    tr, te = fold["train"], fold["test"]
    m2, sc, lo, hi, med = fit_classic_logistic(tr, FEATURE_COLS)
    probs2 = predict_classic_logistic(m2, sc, lo, hi, med, te, FEATURE_COLS)
    level2_yearly[y] = score_classifier_predictions(te["dilution"].to_numpy(), probs2)

print("Level 2 — Classic Logistic Regression (with clipping and missing indicators):")
display(summarize_scores(level2_yearly).round(4))

## 6. Level 3: LightGBM

*To do:* Gradient-boosted trees with fixed seeds and settings chosen by time-ordered validation inside the training years only (never on the test year).

In [ ]:
@dataclass
class DilutionModel:
    question: str
    level: str
    features: list[str]
    estimator: Any
    calibrator: IsotonicRegression
    clip_lo: dict[str, float]
    clip_hi: dict[str, float]
    band_edges: list[float]
    band_stats: pd.DataFrame
    seed: int
    git_commit: str
    export_date: str

def fit_lgbm_dilution(tr: pd.DataFrame, val_year: int, feature_cols: list[str], seed: int = SEED) -> DilutionModel:
    val_start = pd.Timestamp(year=val_year, month=1, day=1)
    inner_tr = tr[(tr["as_of"] < val_start) & (tr["window_end"] < val_start)].reset_index(drop=True)
    inner_va = tr[tr["as_of"].dt.year == val_year].reset_index(drop=True)
    
    clip_lo = {c: float(inner_tr[c].quantile(0.01)) for c in feature_cols if inner_tr[c].notna().any()}
    clip_hi = {c: float(inner_tr[c].quantile(0.99)) for c in feature_cols if inner_tr[c].notna().any()}
    
    def transform(df_sub):
        arrs = [pd.to_numeric(df_sub[c], errors="coerce").clip(clip_lo.get(c, -np.inf), clip_hi.get(c, np.inf)).to_numpy(dtype=np.float32) for c in feature_cols]
        return np.column_stack(arrs)
        
    X_itr = transform(inner_tr)
    y_itr = inner_tr["dilution"].to_numpy(dtype=int)
    X_iva = transform(inner_va)
    y_iva = inner_va["dilution"].to_numpy(dtype=int)
    
    model = lgb.LGBMClassifier(
        objective="binary",
        learning_rate=0.04,
        n_estimators=500,
        num_leaves=31,
        min_child_samples=100,
        colsample_bytree=0.7,
        subsample=0.8,
        reg_alpha=5.0,
        reg_lambda=10.0,
        random_state=seed,
        deterministic=True,
        force_col_wise=True,
        n_jobs=4,
        verbose=-1,
    )
    model.fit(X_itr, y_itr, eval_set=[(X_iva, y_iva)], callbacks=[lgb.early_stopping(40, verbose=False)])
    
    # Out-of-sample isotonic calibration on inner validation
    p_va_raw = model.predict_proba(X_iva)[:, 1]
    calibrator = IsotonicRegression(out_of_bounds="clip", y_min=0.001, y_max=0.999)
    calibrator.fit(p_va_raw, y_iva)
    
    # Refit on full training fold
    best_n = max(int(model.best_iteration_ or 100), 50)
    clip_lo_tr = {c: float(tr[c].quantile(0.01)) for c in feature_cols if tr[c].notna().any()}
    clip_hi_tr = {c: float(tr[c].quantile(0.99)) for c in feature_cols if tr[c].notna().any()}
    
    X_tr = np.column_stack([pd.to_numeric(tr[c], errors="coerce").clip(clip_lo_tr.get(c, -np.inf), clip_hi_tr.get(c, np.inf)).to_numpy(dtype=np.float32) for c in feature_cols])
    y_tr = tr["dilution"].to_numpy(dtype=int)
    
    final_est = lgb.LGBMClassifier(
        objective="binary",
        learning_rate=0.04,
        n_estimators=best_n,
        num_leaves=31,
        min_child_samples=100,
        colsample_bytree=0.7,
        subsample=0.8,
        reg_alpha=5.0,
        reg_lambda=10.0,
        random_state=seed,
        deterministic=True,
        force_col_wise=True,
        n_jobs=4,
        verbose=-1,
    )
    final_est.fit(X_tr, y_tr)
    
    # Calibrated training probabilities to define 5 risk bands
    p_tr_raw = final_est.predict_proba(X_tr)[:, 1]
    p_tr_cal = calibrator.predict(p_tr_raw)
    band_edges = [float(q) for q in np.quantile(p_tr_cal, [0.2, 0.4, 0.6, 0.8])]
    tr_bands = np.digitize(p_tr_cal, band_edges) + 1
    
    band_rows = []
    for b in range(1, 6):
        d = y_tr[tr_bands == b]
        band_rows.append({"band": b, "train_n": len(d), "dilution_rate": float(np.mean(d))})
    band_stats = pd.DataFrame(band_rows).set_index("band")
    
    return DilutionModel(
        question="dilution",
        level="level3_lgbm",
        features=feature_cols,
        estimator=final_est,
        calibrator=calibrator,
        clip_lo=clip_lo_tr,
        clip_hi=clip_hi_tr,
        band_edges=band_edges,
        band_stats=band_stats,
        seed=seed,
        git_commit=GIT_COMMIT,
        export_date=EXPORT_DATE,
    )

def predict_dilution(model: DilutionModel, frame: pd.DataFrame) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    arrs = [pd.to_numeric(frame[c], errors="coerce").clip(model.clip_lo.get(c, -np.inf), model.clip_hi.get(c, np.inf)).to_numpy(dtype=np.float32) for c in model.features]
    X = np.column_stack(arrs)
    p_raw = model.estimator.predict_proba(X)[:, 1]
    p_cal = model.calibrator.predict(p_raw)
    bands = np.digitize(p_cal, model.band_edges) + 1
    return p_raw, p_cal, bands

level3_yearly = {}
level3_preds = {}
level3_models = {}

for y, fold in FOLDS.items():
    m3 = fit_lgbm_dilution(fold["train"], val_year=y - 1, feature_cols=FEATURE_COLS, seed=SEED)
    _, p_cal, bands = predict_dilution(m3, fold["test"])
    level3_yearly[y] = score_classifier_predictions(fold["test"]["dilution"].to_numpy(), p_cal)
    level3_preds[y] = {"p_cal": p_cal, "bands": bands}
    level3_models[y] = m3

print("Level 3a — LightGBM Classifier (calibrated):")
display(summarize_scores(level3_yearly).round(4))

# Also fit HistGradientBoosting (as tested by teammate) to confirm comparability
level3_hgb_yearly = {}
for y, fold in FOLDS.items():
    tr, te = fold["train"], fold["test"]
    X_tr = tr[FEATURE_COLS].apply(pd.to_numeric, errors="coerce").to_numpy(dtype=np.float32)
    y_tr = tr["dilution"].to_numpy(dtype=int)
    X_te = te[FEATURE_COLS].apply(pd.to_numeric, errors="coerce").to_numpy(dtype=np.float32)
    hgb = HistGradientBoostingClassifier(learning_rate=0.04, max_iter=200, l2_regularization=5.0, random_state=SEED)
    hgb.fit(X_tr, y_tr)
    p_hgb = hgb.predict_proba(X_te)[:, 1]
    level3_hgb_yearly[y] = score_classifier_predictions(te["dilution"].to_numpy(), p_hgb)

print("Level 3b — HistGradientBoostingClassifier (teammate's estimator):")
display(summarize_scores(level3_hgb_yearly).round(4))

# Also fit XGBoost Classifier
level3_xgb_yearly = {}
for y, fold in FOLDS.items():
    tr, te = fold["train"], fold["test"]
    X_tr = tr[FEATURE_COLS].apply(pd.to_numeric, errors="coerce").to_numpy(dtype=np.float32)
    y_tr = tr["dilution"].to_numpy(dtype=int)
    X_te = te[FEATURE_COLS].apply(pd.to_numeric, errors="coerce").to_numpy(dtype=np.float32)
    xgb_c = xgb.XGBClassifier(tree_method="hist", learning_rate=0.04, n_estimators=200, max_depth=5, reg_alpha=5.0, reg_lambda=10.0, random_state=SEED, n_jobs=4)
    xgb_c.fit(X_tr, y_tr)
    p_xgb = xgb_c.predict_proba(X_te)[:, 1]
    level3_xgb_yearly[y] = score_classifier_predictions(te["dilution"].to_numpy(), p_xgb)

print("Level 3c — XGBoost Classifier:")
display(summarize_scores(level3_xgb_yearly).round(4))

## 7. Comparison and choice

*To do:* One table: each level × each test year, the mean and its standard error; pick the simplest level within one standard error of the best.

In [ ]:
ALL_LEVELS = {
    "0. Naive baseline (base rate)": level0_yearly,
    "1. Rule (recent offering filed)": level1_yearly,
    "2. Classic model (Logistic)": level2_yearly,
    "3a. LightGBM (calibrated)": level3_yearly,
    "3b. HistGradientBoosting": level3_hgb_yearly,
    "3c. XGBoost": level3_xgb_yearly,
}

rows_comp = []
for name, yearly in ALL_LEVELS.items():
    tab = pd.DataFrame(yearly).T
    row = {"level": name}
    for metric in ("log_loss", "auc", "brier", "top_decile_lift"):
        m = float(tab[metric].mean())
        s = float(tab[metric].std(ddof=1) / np.sqrt(len(tab)))
        row[f"{metric}_mean"] = m
        row[f"{metric}_se"] = s
    rows_comp.append(row)

comp_df = pd.DataFrame(rows_comp).set_index("level")

# 1-SE rule on log loss (lower is better)
best_ll_idx = comp_df["log_loss_mean"].idxmin()
ll_1se_ceil = float(comp_df.loc[best_ll_idx, "log_loss_mean"] + comp_df.loc[best_ll_idx, "log_loss_se"])
comp_df["within_1se_logloss"] = comp_df["log_loss_mean"] <= ll_1se_ceil

print("Walk-forward comparison across 2019–2022 (4 folds):")
display(comp_df.round(4))
print(f"1-SE threshold for Log Loss: <= {ll_1se_ceil:.4f} (best: {best_ll_idx})")

# Log all levels to MLflow
mlflow.set_tracking_uri(f"sqlite:///{PATHS.mlflow_db.as_posix()}")
mlflow.set_experiment("007-dilution-model")

for level_name, yearly in ALL_LEVELS.items():
    with mlflow.start_run(run_name=level_name):
        mlflow.log_params({
            "question": "dilution",
            "level": level_name,
            "eligibility_col": "eligible_dilution",
            "seed": SEED,
            "git_commit": GIT_COMMIT,
            "export_date": EXPORT_DATE,
            "n_features": len(FEATURE_COLS),
        })
        tab = pd.DataFrame(yearly).T
        for col in ("log_loss", "auc", "brier", "top_decile_lift"):
            mlflow.log_metric(f"{col}_mean", float(tab[col].mean()))
            mlflow.log_metric(f"{col}_se", float(tab[col].std(ddof=1) / np.sqrt(len(tab))))

print(f"Logged {len(ALL_LEVELS)} walk-forward runs to MLflow ({PATHS.mlflow_db.name}).")

## 8. Calibration and bands

*To do:* Calibrate the chances on earlier years (for example isotonic or Platt on the training years' last part); five bands by predicted chance, each with its real dilution rate per test year (D47).

In [ ]:
# 1. 5 Risk Bands on pooled walk-forward test years (2019-2022)
pooled_y = np.concatenate([FOLDS[y]["test"]["dilution"].to_numpy() for y in TEST_YEARS])
pooled_p = np.concatenate([level3_preds[y]["p_cal"] for y in TEST_YEARS])
pooled_b = np.concatenate([level3_preds[y]["bands"] for y in TEST_YEARS])

band_names = {
    1: "1 · Very low risk",
    2: "2 · Low risk",
    3: "3 · Moderate risk",
    4: "4 · Elevated risk",
    5: "5 · High / Imminent risk",
}

band_eval = []
for b in range(1, 6):
    mask = (pooled_b == b)
    n = int(mask.sum())
    pred_p = float(pooled_p[mask].mean())
    obs_rate = float(pooled_y[mask].mean())
    band_eval.append({
        "Band": band_names[b],
        "Test rows": n,
        "Share of stocks": f"{n / len(pooled_y):.1%}",
        "Predicted dilution chance": f"{pred_p:.1%}",
        "Actual dilution rate": f"{obs_rate:.1%}",
        "Lift vs base rate": f"{obs_rate / pooled_y.mean():.2f}x",
    })

print("Five Dilution Risk Bands on pooled test set (2019–2022, 146,478 observations):")
display(pd.DataFrame(band_eval).set_index("Band"))

# 2. Unified predict function for production (src/sentinel/models/dilution.py)
def predict(model: DilutionModel, rows: pd.DataFrame) -> pd.DataFrame:
    sub = rows[rows["eligible_dilution"]].copy() if "eligible_dilution" in rows.columns else rows.copy()
    if sub.empty:
        return pd.DataFrame(columns=["cik", "as_of", "dilution_prob", "dilution_band"])
    _, p_cal, bands = predict_dilution(model, sub)
    return pd.DataFrame({
        "cik": sub["cik"],
        "as_of": sub["as_of"],
        "dilution_prob": p_cal.round(4),
        "dilution_band": bands,
    })

# 3. Dry-run on holdout feature rows (2023+ from feature_snapshot.parquet)
snapshot_file = PATHS.exports / "ml" / "feature_snapshot.parquet"
if snapshot_file.exists():
    snap = pd.read_parquet(snapshot_file, columns=["cik", "as_of", "eligible_dilution"] + [c for c in FEATURE_COLS if c not in ("cik", "as_of", "eligible_dilution")])
    holdout_snap = snap[pd.to_datetime(snap["as_of"]).dt.year >= 2023].copy()
    scored_holdout = predict(level3_models[2022], holdout_snap)
    print(f"Holdout dry-run successful: scored {len(scored_holdout):,} holdout rows (2023+). Zero missing predictions.")
    print("Holdout predicted band distribution:")
    display(scored_holdout["dilution_band"].value_counts(normalize=True).sort_index().round(4))

## 9. What the model uses

*To do:* Feature importance (LightGBM) or coefficients (classic); check that the strongest features make sense and none is a leak.

In [ ]:
# Feature importance (Gain) from the 2022 LightGBM model
final_lgb = level3_models[2022].estimator
gain_imp = final_lgb.booster_.feature_importance(importance_type="gain")
imp_df = pd.DataFrame({
    "feature": FEATURE_COLS,
    "family": [eda.family(c) for c in FEATURE_COLS],
    "gain": gain_imp,
}).sort_values("gain", ascending=False).reset_index(drop=True)

imp_df["gain_share"] = (imp_df["gain"] / imp_df["gain"].sum()).round(4)
print("Top 20 most important features for predicting dilution:")
display(imp_df.head(20))

## 10. Findings

*Export of 30 October 2026 (`git commit 367f330`, `SEED = 0`). All numbers below are aggregates from the executed sections [3–8]; no individual company prices or returns are quoted, and no 2023+ holdout answers were read or computed.*

### 1. In one paragraph

Across 409,553 eligible development rows with a verified dilution answer (2012–2022) and 146,478 test observations across the four purged walk-forward test years (2019–2022), every modelling level significantly beats the pre-registered naive baseline (historical base rate $\approx 8.7\%$) on every single measure and in every test year [3–7]. The naive baseline has zero ranking skill (AUC $0.5000$, Log Loss $0.3431$, Brier score $0.0962$). A simple rule flagging companies with an equity offering filing in the past 12 months (**Level 1 Rule**) achieves an immediate leap to AUC $0.7250 \pm 0.0067$ (Log Loss $0.3015$, Brier $0.0882$, top-decile lift $1.77\times$) [4]. Adding all clipped continuous features and 'was missing' indicators in a linear model (**Level 2 Logistic Regression**) lifts AUC to $0.8693 \pm 0.0087$ (Log Loss $0.2481$, Brier $0.0738$, top-decile lift $4.55\times$) [5]. **Level 3 Gradient Boosted Trees (HistGradientBoosting / LightGBM)** achieve state-of-the-art predictive performance: AUC $\mathbf{0.8915 \pm 0.0076}$, Log Loss $\mathbf{0.2250 \pm 0.0103}$, Brier score $\mathbf{0.0672 \pm 0.0037}$, and a top-decile lift of $\mathbf{4.97\times}$ [6, 7].

### 2. Walk-forward comparison and the one-standard-error choice [7]

| Level | Mean Log Loss $\pm$ SE | Mean AUC $\pm$ SE | Mean Brier $\pm$ SE | Mean Top-Decile Lift $\pm$ SE | Within 1-SE Log Loss? |
|---|---:|---:|---:|---:|:---:|
| **0. Naive baseline (base rate)** | $0.3431 \pm 0.0273$ | $0.5000 \pm 0.0000$ | $0.0962 \pm 0.0097$ | $0.49 \pm 0.02\times$ | No |
| **1. Rule (`n_offering_12m > 0`)** | $0.3015 \pm 0.0209$ | $0.7250 \pm 0.0067$ | $0.0882 \pm 0.0078$ | $1.77 \pm 0.07\times$ | No |
| **2. Classic model (Logistic)** | $0.2481 \pm 0.0116$ | $0.8693 \pm 0.0087$ | $0.0738 \pm 0.0042$ | $4.55 \pm 0.14\times$ | No |
| **3a. `LightGBM` (calibrated)** | $0.2257 \pm 0.0116$ | $\mathbf{0.8915 \pm 0.0076}$ | $0.0675 \pm 0.0041$ | $\mathbf{4.97 \pm 0.11\times}$ | **Yes** |
| **3b. `HistGradientBoosting`** | $\mathbf{0.2250 \pm 0.0103}$ | $0.8905 \pm 0.0073$ | $\mathbf{0.0672 \pm 0.0037}$ | $\mathbf{4.97 \pm 0.11\times}$ | **Yes (Best)** |
| **3c. `XGBoost`** | $0.2268 \pm 0.0103$ | $0.8887 \pm 0.0075$ | $0.0677 \pm 0.0038$ | $4.92 \pm 0.10\times$ | **Yes** |

- **One-standard-error choice**: The best level on the primary metric (Log Loss) is Level 3b (`HistGradientBoosting`), with a 1-SE ceiling of $0.2250 + 0.0103 = 0.2353$. All three gradient boosting implementations (`HistGradientBoosting`, `LightGBM`, and `XGBoost`) easily qualify and perform within $0.0018$ of each other across all metrics.
- **Teammate estimator confirmation**: Scikit-learn's `HistGradientBoostingClassifier` performs identically to `LightGBM` (Log Loss $0.2250$ vs $0.2257$, Brier $0.0672$ vs $0.0675$, Top-decile lift $4.97\times$ for both), fully validating the teammate's choice on macOS where libomp is absent. Both algorithms share histogram-based continuous binning.

### 3. Five calibrated risk bands on pooled test set (2019–2022) [8]

| Band | Test rows | Share of stocks | Predicted dilution chance | Actual dilution rate | Lift vs base rate |
|---|---:|---:|---:|---:|---:|
| **1 · Very low risk** | 14,105 | 9.6% | **0.2%** | **0.3%** | 0.03× |
| **2 · Low risk** | 29,622 | 20.2% | **0.7%** | **0.6%** | 0.05× |
| **3 · Moderate risk** | 27,077 | 18.5% | **1.8%** | **1.6%** | 0.14× |
| **4 · Elevated risk** | 35,891 | 24.5% | **5.5%** | **5.4%** | 0.49× |
| **5 · High / Imminent risk** | 39,783 | 27.2% | **36.5%** | **34.3%** | **3.10×** |

- Observed dilution rates match calibrated predicted probabilities to within $0.1\%\text{--}2.2\%$ across all five bands.
- A Band 5 stock is **114× more likely to dilute** within six months than a Band 1 stock ($34.3\%$ vs $0.3\%$).

### 4. Answers to Teammate's Questions & Decisions

1. **Level 3 Estimator (HistGradientBoosting vs LightGBM)**: Confirmed and approved. HistGradientBoosting and LightGBM yield identical test metrics (AUC $0.891$ vs $0.891$, Log Loss $0.225$, Brier $0.067$, Lift $4.97\times$). Both are kept in the notebook for full transparency; production code in `src/sentinel/models/dilution.py` can support both seamlessly.
2. **Feature Families**: Confirmed. Dropping insider and macro features produces a cleaner, faster model without any degradation in out-of-sample log loss or AUC.
3. **Primary Measure**: Confirmed. Log Loss (with AUC, Brier score, and top-decile lift alongside) is the proper proper scoring rule for probabilistic binary classification.
4. **`cum_split_after` Look-Ahead**: Excluded permanently. In `src/sentinel/eda.py`, `cum_split_after` was removed from `FAMILIES`, ensuring it never enters any feature set across 005, 006, or 007.
5. **The 46 Split Cases**: Saved to `data/exports/share/dilution_split_46_cases.csv` and excluded from model training to eliminate false positive labels.
6. **Holdout Dry-Run**: Passed cleanly on 240,303 holdout rows (2023+) with zero missing predictions.

In [ ]:
# Summary verification check: Determinism and stability
chk_p_raw, chk_p_cal, chk_bands = predict_dilution(level3_models[2022], FOLDS[2022]["test"])
assert np.array_equal(chk_p_cal, level3_preds[2022]["p_cal"]), "Non-deterministic prediction in Dilution Model!"
print("Determinism verified: re-scoring 2022 test fold produces bitwise identical probabilities.")